# Этап B. Сила сохранённой LoRA: 0 / 0.25 / 0.5 / 1.0

**Подготовленный ноутбук; GPU-прогон ещё не выполнен. Новая сессия Colab, T4 GPU.**
Обучения нет. База и данные закреплены на ревизиях исходного пилота.
Свой ZIP загрузите в ячейке 3 через интерфейс Colab либо Google Drive.
Ожидается адаптер с SHA-256 `461961f38e7552aa6d4c6990b5c1aaffea5801d24b1a4d4a4b86fda61feb98f8`.

Основной результат: **32 sampling-ответа**, по 8 на каждый масштаб.
Отдельно: **12 greedy-контролей** с лимитом 64 токена; ещё 6 коротких
эталонных контролей используются только в технических проверках.
Восемь вопросов выбраны после обнаружения ошибок — это development,
а не независимый финальный тест. Технический PASS не означает улучшение качества.

Запускайте ячейки 1–7 по порядку. Если ячейка прервана, ячейка 7 позволяет
скачать сохранённый частичный результат. Не запускайте параллельно несколько
копий цикла. Для возобновления в текущей сессии повторите ячейку 6.


## 1. Фиксированные версии
Установка соответствует записанному окружению пилота, включая PyTorch/CUDA.
Если уже импортированные библиотеки пришлось обновить, перезапустите сеанс
Python и повторите ячейку. Ошибка установки не является результатом диагностики:
при ней остановитесь и сохраните текст ошибки, не подменяя версии автоматически.


In [ ]:
import importlib.metadata
import os
import subprocess
import sys
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
PINS = {'transformers': '5.22.1', 'tokenizers': '0.23.3', 'huggingface-hub': '2.4.0', 'accelerate': '1.12.0', 'bitsandbytes': '0.50.2', 'peft': '0.21.2', 'safetensors': '0.8.0'}
TORCH_PIN = '2.14.1+cu130'
def installed(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
changed = {p for p,v in PINS.items() if installed(p) != v}
if installed('torch') != TORCH_PIN:
    changed.add('torch')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'torch=='+TORCH_PIN, '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
if changed & set(PINS):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    *[p+'=='+v for p,v in PINS.items()]], check=True)
expected_versions = dict(PINS, torch=TORCH_PIN)
loaded = [p for p,v in expected_versions.items()
          if p.replace('-', '_') in sys.modules
          and (p in changed or getattr(sys.modules[p.replace('-', '_')], '__version__', v) != v)]
if loaded:
    raise RuntimeError('Перезапустите сеанс Python после установки: '+', '.join(loaded))
assert installed('torch') == TORCH_PIN
assert all(installed(p)==v for p,v in PINS.items())
print('Версии установлены. GPU-диагностика ещё не запускалась.')


## 2. Данные из закреплённого коммита и отдельный каталог результатов
`OUTPUT_PARENT` можно изменить на каталог Google Drive после его подключения.
По умолчанию результаты находятся в `/content`; скачайте ZIP до окончания сеанса.
Все исходные файлы сверяются по SHA-256. Seed берётся из позиции вопроса
во **всех 32** вопросах, а не из позиции в диагностической восьмёрке.


In [ ]:
import contextlib, hashlib, inspect, json, math, platform, random, re, shutil, time, traceback, urllib.request, zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
import peft
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig, set_seed
from peft import PeftModel, prepare_model_for_kbit_training
from peft.tuners.lora.layer import LoraLayer

if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Выберите в Colab именно T4 GPU.')
if 'model' in globals() or 'RUN_DIR' in globals():
    raise RuntimeError('Ячейка инициализации уже выполнена. Для продолжения используйте ячейки 6–7; для нового прогона создайте новую сессию.')
assert torch.__version__ == TORCH_PIN and torch.version.cuda == '13.0'
assert peft.__version__ == PINS['peft']
assert all(getattr(importlib.import_module(p.replace('-', '_')), '__version__', v)==v for p,v in PINS.items())
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
COMMIT = '89dd84aaf40feac11563802b100a97076fb9b72e'
SOURCE_HASHES = {'data/sft_v3/selection.json': '1a4e9e6b91452d75ce54771ebe96003afd92c13839de9eb5df44f32d1874e5e4', 'data/sft_v3/pilot_config.json': 'b7fc10b036918bb0b92b8ace8f4a014e6eb90b04672cd41e9efcad575cfc7383', 'data/development_v3/questions.jsonl': '440228aa8ed0a2dff6c3e59b4be5155398e6c0937975fedf0b59312f5253d629', 'data/development_v3/rubric.jsonl': 'dfb1f18076e2ea7b28ce0926225fe34efb5e63b9e8ae0956aa0e93f9b8897ae7', 'data/development_v3/sources.json': '993619d62e59d60753c4f411dd3e537290b64f024c282565d78f16c8e730d88d', 'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/answers.jsonl': 'e686d96d5a08692185b1687e04242abe3a4d1d1c06356ba5ff0439f4d42d01f3', 'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/adapter/adapter_config.json': '2243c4784b72906e97b4c04afb0f671a51972625fe07e76ce949fa83ebf77b92'}
SOURCE_ROOT = Path('/content/lm_hw_strength_source_'+COMMIT[:7])
OUTPUT_PARENT = Path('/content/lm_hw_strength_runs')
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
for relative, expected in SOURCE_HASHES.items():
    dest = SOURCE_ROOT/relative
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists() or hashlib.sha256(dest.read_bytes()).hexdigest()!=expected:
        url = 'https://raw.githubusercontent.com/maria-igntv/lm-hw/'+COMMIT+'/'+relative
        with urllib.request.urlopen(url, timeout=120) as response: data = response.read()
        if hashlib.sha256(data).hexdigest()!=expected: raise ValueError('SHA mismatch: '+relative)
        dest.write_bytes(data)
selection = json.loads((SOURCE_ROOT/'data/sft_v3/selection.json').read_text())
pilot_settings = json.loads((SOURCE_ROOT/'data/sft_v3/pilot_config.json').read_text())
questions = [json.loads(s) for s in (SOURCE_ROOT/'data/development_v3/questions.jsonl').read_text().splitlines() if s.strip()]
assert len(questions)==32 and len({q['id'] for q in questions})==32
question_index = {q['id']:q for q in questions}
SEEDS = {q['id']:20261116+i for i,q in enumerate(questions,1)}
DIAGNOSTIC_IDS = ['dev3_01','dev3_07','dev3_15','dev3_21','dev3_24','dev3_25','dev3_26','dev3_29']
GREEDY_IDS = ['dev3_01','dev3_07','dev3_25']
STRENGTHS = [0.0,0.25,0.5,1.0]
GENERATION = dict(pilot_settings['generation'])
assert GENERATION == {'max_new_tokens':768,'do_sample':True,'temperature':0.7,'top_p':0.8,'top_k':20,'min_p':0.0,'repetition_penalty':1.0,'num_beams':1,'use_cache':True}
assert selection['model_id']=='Qwen/Qwen3-4B-Instruct-2507'
assert selection['revision']=='cdbee75f17c01a7cc42f958dc650907174af0554'
RUN_ID = 'adapter_strength_v1_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
RUN_DIR = OUTPUT_PARENT/RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
def write_json(path, value):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n'); tmp.replace(path)
def save_rows(path, rows):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in rows));tmp.replace(path)
def read_rows(path):
    return [json.loads(s) for s in Path(path).read_text().splitlines() if s.strip()] if Path(path).exists() else []
def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as fh:
        for block in iter(lambda:fh.read(8<<20),b''):h.update(block)
    return h.hexdigest()
metadata = {'run_id':RUN_ID,'status':'prepared_not_executed','source_commit':COMMIT,
 'started_at_utc':datetime.now(timezone.utc).isoformat(),'selection':selection,
 'generation':GENERATION,'strengths':STRENGTHS,'seeds':{q:SEEDS[q] for q in DIAGNOSTIC_IDS},
 'diagnostic_ids':DIAGNOSTIC_IDS,'greedy_ids':GREEDY_IDS,'source_hashes':SOURCE_HASHES,
 'versions':{p:installed(p) for p in ['torch',*PINS,'numpy']},
 'hardware':{'gpu':torch.cuda.get_device_name(0),'python':platform.python_version(),'cuda':torch.version.cuda},
 'scope':'post-hoc diagnostic development; no training; no independent-test claim'}
write_json(RUN_DIR/'run_metadata.json',metadata)
for relative in SOURCE_HASHES:
    dest=RUN_DIR/'source'/relative;dest.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(SOURCE_ROOT/relative,dest)
print('Каталог:',RUN_DIR)
print('Seeds:',metadata['seeds'])


## 3. Загружаем пользовательский ZIP и проверяем адаптер
Вариант `upload` открывает загрузку Colab. Вариант `drive` подключает Drive;
в `DRIVE_ZIP` укажите путь внутри подключённого Drive. Путей с вашего компьютера
здесь нет. Из ZIP читаются только два файла адаптера, код из архива не выполняется.


In [ ]:
ZIP_SOURCE = 'upload'  # 'upload' или 'drive'
DRIVE_ZIP = '/content/drive/MyDrive/sft_v3_pilot_qwen3_4b_20261118T113449101907.zip'
EXPECTED_ADAPTER_SHA = '461961f38e7552aa6d4c6990b5c1aaffea5801d24b1a4d4a4b86fda61feb98f8'
if ZIP_SOURCE=='drive':
    from google.colab import drive
    drive.mount('/content/drive')
    USER_ZIP = Path(DRIVE_ZIP)
elif ZIP_SOURCE=='upload':
    from google.colab import files
    uploaded = files.upload()
    names=[n for n in uploaded if n.lower().endswith('.zip')]
    if len(names)!=1: raise ValueError('Загрузите ровно один ZIP исходного пилота.')
    USER_ZIP=Path('/content')/names[0]
    # files.upload normally writes the file into the current directory.
    if not USER_ZIP.exists(): USER_ZIP.write_bytes(uploaded[names[0]])
    del uploaded
else: raise ValueError('ZIP_SOURCE должен быть upload или drive')
assert USER_ZIP.is_file()
ADAPTER_DIR=Path('/content/lm_hw_strength_inputs')/RUN_ID/'adapter'
ADAPTER_DIR.mkdir(parents=True,exist_ok=False)
prefix='qwen3_4b_20261118T113449101907/adapter/'
with zipfile.ZipFile(USER_ZIP) as archive:
    for name in ['adapter_config.json','adapter_model.safetensors']:
        info=archive.getinfo(prefix+name)
        if info.file_size > 1_000_000_000: raise ValueError('Неожиданный размер файла адаптера')
        with archive.open(info) as source, (ADAPTER_DIR/name).open('wb') as dest:
            shutil.copyfileobj(source,dest,8<<20)
actual=sha_file(ADAPTER_DIR/'adapter_model.safetensors')
if actual!=EXPECTED_ADAPTER_SHA: raise ValueError('SHA-256 адаптера не совпал; модель не загружается.')
adapter_config=json.loads((ADAPTER_DIR/'adapter_config.json').read_text())
public_config=json.loads((SOURCE_ROOT/'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/adapter/adapter_config.json').read_text())
if adapter_config!=public_config: raise ValueError('Конфигурация адаптера отличается от опубликованной.')
assert adapter_config['r']==16 and adapter_config['lora_alpha']==32 and adapter_config['lora_dropout']==0
assert not adapter_config['use_rslora'] and not adapter_config['use_dora']
metadata.update(adapter_sha256=actual,user_zip_sha256=sha_file(USER_ZIP))
write_json(RUN_DIR/'adapter_config.json',adapter_config)
write_json(RUN_DIR/'run_metadata.json',metadata)
print('SHA адаптера подтвержден:',actual)


## 4. Закреплённая база NF4/FP16/eager и сохранённая LoRA
Воспроизводим `prepare_model_for_kbit_training` пилота, но веса заморожены:
оптимизатора, backward и обучения в этом ноутбуке нет.


In [ ]:
set_seed(pilot_settings['seed'])
tokenizer=AutoTokenizer.from_pretrained(selection['model_id'],revision=selection['revision'],trust_remote_code=False)
tokenizer.padding_side='right'
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
quant=dict(pilot_settings['quantization']);quant['bnb_4bit_compute_dtype']=torch.float16
base_model=AutoModelForCausalLM.from_pretrained(
 selection['model_id'],revision=selection['revision'],trust_remote_code=False,
 dtype=torch.float16,device_map={'':0},quantization_config=BitsAndBytesConfig(**quant),
 attn_implementation='eager').eval()
assert base_model.__class__.__name__=='Qwen3ForCausalLM'
assert getattr(base_model,'is_loaded_in_4bit',False)
eos_ids=base_model.generation_config.eos_token_id
if isinstance(eos_ids,int): eos_ids=[eos_ids]
assert set(eos_ids)=={151645,151643}
base_model=prepare_model_for_kbit_training(base_model,use_gradient_checkpointing=True,
 gradient_checkpointing_kwargs={'use_reentrant':False})
base_model.config.use_cache=False
model=PeftModel.from_pretrained(base_model,str(ADAPTER_DIR),adapter_name='default',
 is_trainable=False,autocast_adapter_dtype=True).eval()
del base_model
assert set(model.peft_config)=={'default'}
assert not any(p.requires_grad for p in model.parameters())
assert all(p.dtype==torch.float32 for n,p in model.named_parameters() if 'lora_' in n)
assert all(p.device.type=='cuda' for p in model.parameters())
generation_config=GenerationConfig(**GENERATION,eos_token_id=eos_ids,pad_token_id=tokenizer.pad_token_id)
greedy_config=GenerationConfig(do_sample=False,max_new_tokens=64,use_cache=True,
 eos_token_id=eos_ids,pad_token_id=tokenizer.pad_token_id)
canonical={}
recorded=[json.loads(s) for s in (SOURCE_ROOT/'results/sft_v3_pilot/qwen3_4b_20261118T113449101907/answers.jsonl').read_text().splitlines() if s.strip()]
baseline_records={r['id']:r for r in recorded if r['phase']=='baseline'}
for qid in DIAGNOSTIC_IDS:
    encoded=tokenizer.apply_chat_template([{'role':'system','content':selection['system_prompt']},
     {'role':'user','content':question_index[qid]['question']}],add_generation_prompt=True,
     tokenize=True,return_dict=True,return_tensors='pt')
    canonical[qid]={k:v.cpu().clone() for k,v in encoded.items()}
    assert encoded['input_ids'][0].tolist()==baseline_records[qid]['input_token_ids'], qid
    assert SEEDS[qid]==baseline_records[qid]['seed'], qid
    assert encoded['input_ids'].shape[1]+768<=model.config.max_position_embeddings
write_json(RUN_DIR/'input_audit.json',{qid:{'seed':SEEDS[qid],
 'input_token_ids':canonical[qid]['input_ids'][0].tolist(),
 'rendered_prompt':tokenizer.decode(canonical[qid]['input_ids'][0].tolist(),skip_special_tokens=False)} for qid in DIAGNOSTIC_IDS})
print('База и LoRA загружены. Все 8 входов и seed совпали с исходным пилотом.')


## 5. Абсолютный масштаб, контроль весов и RNG
В установленной PEFT проверяются версия, сигнатура и фактические масштабы всех
252 LoRA-слоёв. Используется `set_scale`, а не накопительное `scale_layer`.
Каждая генерация сохраняет и восстанавливает Python/NumPy/CPU/CUDA RNG.


In [ ]:
layers={n:m for n,m in model.named_modules() if isinstance(m,LoraLayer) and 'default' in m.scaling}
assert len(layers)==252
original_scales={n:float(m.scaling['default']) for n,m in layers.items()}
for n,layer in layers.items():
    assert not layer.merged and layer.r['default']==16 and layer.lora_alpha['default']==32
    assert math.isclose(original_scales[n],2.0)
    assert tuple(inspect.signature(layer.set_scale).parameters)==('adapter','scale')
method_source=inspect.getsource(next(iter(layers.values())).set_scale)
write_json(RUN_DIR/'peft_scale_api.json',{'peft_version':peft.__version__,
 'method':'LoraLayer.set_scale(adapter, scale)', 'source_sha256':hashlib.sha256(method_source.encode()).hexdigest(),
 'reference':'https://github.com/huggingface/peft/blob/v0.21.2/src/peft/tuners/lora/layer.py',
 'layer_classes':sorted({type(x).__module__+'.'+type(x).__name__ for x in layers.values()}),
 'n_layers':len(layers),'initial_scaling':original_scales})
def apply_strength(multiplier):
    if multiplier not in STRENGTHS: raise ValueError('Неизвестный масштаб')
    for name,layer in layers.items():
        layer.set_scale('default',float(multiplier))
        if not math.isclose(float(layer.scaling['default']),original_scales[name]*multiplier,rel_tol=1e-12,abs_tol=1e-12):
            raise RuntimeError('API масштаба не соответствует alpha/r: '+name)
    return multiplier
@contextlib.contextmanager
def isolated_rng(seed):
    py=random.getstate();np_state=np.random.get_state();cpu=torch.random.get_rng_state();cuda=torch.cuda.get_rng_state_all()
    try:
        set_seed(seed)
        yield
    finally:
        random.setstate(py);np.random.set_state(np_state);torch.random.set_rng_state(cpu);torch.cuda.set_rng_state_all(cuda)
def input_batch(qid):
    return {k:v.clone().to('cuda') for k,v in canonical[qid].items()}
def parameter_fingerprint():
    # Stream small byte slices to CPU instead of copying the whole 4B model.
    result={}
    for name,param in model.named_parameters():
        h=hashlib.sha256();h.update(str((tuple(param.shape),str(param.dtype))).encode())
        raw=param.detach().contiguous().reshape(-1).view(torch.uint8).reshape(-1)
        for start in range(0,raw.numel(),4<<20):h.update(raw[start:start+(4<<20)].cpu().numpy().tobytes())
        result[name]=h.hexdigest()
    return result
@torch.inference_mode()
def prompt_logits(qid):
    model.eval()
    batch=input_batch(qid)
    result=model(**batch,use_cache=False,logits_to_keep=1).logits[0,-1].float().cpu()
    if not torch.isfinite(result).all():raise FloatingPointError('Logits NaN/Inf')
    return result
@torch.inference_mode()
def generate_record(qid,strength,kind):
    config=generation_config if kind=='sampling' else greedy_config
    was_training=model.training;previous_cache=model.config.use_cache
    try:
        model.eval();model.config.use_cache=True
        with isolated_rng(SEEDS[qid]):
            batch=input_batch(qid);prefix=batch['input_ids'].shape[1]
            assert batch['input_ids'][0].tolist()==canonical[qid]['input_ids'][0].tolist()
            torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();started=time.perf_counter()
            output=model.generate(**batch,generation_config=config)
            torch.cuda.synchronize();elapsed=time.perf_counter()-started
            ids=output[0,prefix:].tolist()
            finish='eos' if ids and ids[-1] in eos_ids else ('length' if len(ids)>=config.max_new_tokens else 'other')
            return {'question_id':qid,'category':question_index[qid]['category'],
             'dangerous':question_index[qid]['dangerous'],'question':question_index[qid]['question'],
             'adapter_multiplier':strength,'generation_kind':kind,'seed':SEEDS[qid],
             'answer':tokenizer.decode(ids,skip_special_tokens=True),'raw_answer':tokenizer.decode(ids,skip_special_tokens=False),
             'generated_token_ids':ids,'generated_tokens':len(ids),'finish_reason':finish,
             'elapsed_seconds':elapsed,'peak_allocated_gib':torch.cuda.max_memory_allocated()/1024**3,
             'peak_reserved_gib':torch.cuda.max_memory_reserved()/1024**3,
             'input_token_ids':batch['input_ids'][0].tolist(),'max_new_tokens':config.max_new_tokens}
    finally:
        model.config.use_cache=previous_cache;model.train(was_training)

apply_strength(1.0)
reference_logits={q:prompt_logits(q) for q in DIAGNOSTIC_IDS}
# Finish lazy inference initialization before the invariant parameter snapshot.
weights_before=parameter_fingerprint()
write_json(RUN_DIR/'parameter_hashes_before.json',weights_before)
scale1_reference={q:generate_record(q,1.0,'greedy') for q in GREEDY_IDS}
save_rows(RUN_DIR/'scale1_reference_controls.jsonl',list(scale1_reference.values()))
for s in [0.5,0.25,0.5,1.0,0.0,1.0]:apply_strength(s)
cycle_check={q:float((prompt_logits(q)-reference_logits[q]).abs().max()) for q in DIAGNOSTIC_IDS}
assert all(v==0 for v in cycle_check.values()), cycle_check
apply_strength(0.0)
zero_logits={q:prompt_logits(q) for q in DIAGNOSTIC_IDS}
zero_check={}
disabled_reference={}
with model.disable_adapter():
    for q in DIAGNOSTIC_IDS:
        disabled=prompt_logits(q)
        zero_check[q]={'max_abs_diff':float((zero_logits[q]-disabled).abs().max()),
          'allclose':bool(torch.allclose(zero_logits[q],disabled,rtol=1e-5,atol=1e-5))}
    for q in GREEDY_IDS:disabled_reference[q]=generate_record(q,0.0,'greedy')
assert all(v['allclose'] for v in zero_check.values()), zero_check
save_rows(RUN_DIR/'disabled_reference_controls.jsonl',list(disabled_reference.values()))
apply_strength(1.0)
verification={'zero_vs_disabled_logits':zero_check,'absolute_scale_cycle_max_abs_diff':cycle_check,
 'zero_vs_disabled_greedy':{},'one_restored_greedy':{},'one_restored_logits':{},
 'weights_unchanged':None,'status':'initial_checks_passed_not_complete'}
write_json(RUN_DIR/'verification.json',verification)
metadata['status']='initial_checks_passed';write_json(RUN_DIR/'run_metadata.json',metadata)
print('Начальные проверки прошли. Основные 32 ответа ещё не сгенерированы.')


## 6. Диагностика: 32 основных ответа + 12 отдельных greedy-контролей
После **каждого вопроса** JSONL атомарно сохраняется. Прерванную ячейку можно
повторить в текущей сессии: уже сохранённые записи проверяются и пропускаются.
При изменении условий создайте новую сессию и новый каталог.


In [ ]:
def ensure_record(row,qid,strength,kind):
    assert row['question_id']==qid and row['adapter_multiplier']==strength and row['generation_kind']==kind
    assert row['seed']==SEEDS[qid] and row['input_token_ids']==canonical[qid]['input_ids'][0].tolist()
    assert row['generated_tokens']==len(row['generated_token_ids'])
    assert row['max_new_tokens']==(768 if kind=='sampling' else 64)
    assert row['answer']==tokenizer.decode(row['generated_token_ids'],skip_special_tokens=True)
def run_diagnostic():
    answers=read_rows(RUN_DIR/'answers.jsonl');controls=read_rows(RUN_DIR/'greedy_controls.jsonl')
    for data,kind in [(answers,'sampling'),(controls,'greedy')]:
        keys=[(r['adapter_multiplier'],r['question_id']) for r in data]
        assert len(keys)==len(set(keys))
        allowed=DIAGNOSTIC_IDS if kind=='sampling' else GREEDY_IDS
        for r in data:
            assert r['adapter_multiplier'] in STRENGTHS and r['question_id'] in allowed
            ensure_record(r,r['question_id'],r['adapter_multiplier'],kind)
    metadata['status']='running';write_json(RUN_DIR/'run_metadata.json',metadata)
    try:
        for strength in STRENGTHS:
            apply_strength(strength)
            for qid in DIAGNOSTIC_IDS:
                if not any(r['adapter_multiplier']==strength and r['question_id']==qid for r in answers):
                    row=generate_record(qid,strength,'sampling');ensure_record(row,qid,strength,'sampling')
                    answers.append(row);save_rows(RUN_DIR/'answers.jsonl',answers)
                    print('sampling',len(answers),'/32',strength,qid,row['finish_reason'],flush=True)
            for qid in GREEDY_IDS:
                if not any(r['adapter_multiplier']==strength and r['question_id']==qid for r in controls):
                    row=generate_record(qid,strength,'greedy');ensure_record(row,qid,strength,'greedy')
                    controls.append(row);save_rows(RUN_DIR/'greedy_controls.jsonl',controls)
                    print('greedy',len(controls),'/12',strength,qid,row['finish_reason'],flush=True)
        assert len(answers)==32 and len(controls)==12
        for qid in GREEDY_IDS:
            zero=next(r for r in controls if r['adapter_multiplier']==0 and r['question_id']==qid)
            one=next(r for r in controls if r['adapter_multiplier']==1 and r['question_id']==qid)
            verification['zero_vs_disabled_greedy'][qid]=zero['generated_token_ids']==disabled_reference[qid]['generated_token_ids']
            verification['one_restored_greedy'][qid]=one['generated_token_ids']==scale1_reference[qid]['generated_token_ids']
        apply_strength(1.0)
        for qid in DIAGNOSTIC_IDS:
            verification['one_restored_logits'][qid]=bool(torch.equal(prompt_logits(qid),reference_logits[qid]))
        weights_after=parameter_fingerprint();write_json(RUN_DIR/'parameter_hashes_after.json',weights_after)
        verification['weights_unchanged']=weights_before==weights_after
        assert all(verification['zero_vs_disabled_greedy'].values())
        assert all(verification['one_restored_greedy'].values())
        assert all(verification['one_restored_logits'].values())
        assert verification['weights_unchanged'] and sha_file(ADAPTER_DIR/'adapter_model.safetensors')==EXPECTED_ADAPTER_SHA
        verification['status']='TECHNICAL_PASS_NOT_QUALITY_CLAIM'
        metadata.update(status='complete_awaiting_content_review',completed_at_utc=datetime.now(timezone.utc).isoformat(),main_answers=32,greedy_controls=12)
        print('TECHNICAL PASS. Содержательное улучшение не установлено.')
    except BaseException as exc:
        metadata['status']='interrupted_or_failed'
        save_rows(RUN_DIR/'errors.jsonl',read_rows(RUN_DIR/'errors.jsonl')+[{'type':type(exc).__name__,'message':str(exc),'traceback':traceback.format_exc()}])
        raise
    finally:
        apply_strength(1.0)
        write_json(RUN_DIR/'verification.json',verification)
        write_json(RUN_DIR/'run_metadata.json',metadata)
run_diagnostic()


## 7. Отчёт и скачиваемый ZIP, включая частичный результат
Отчёт содержит технические индикаторы и полные тексты. Они не заменяют
проверку фактов, безопасности и независимого судью. После скачивания передайте
ZIP для содержательного разбора; только затем выбирается дизайн этапа C.


In [ ]:
def repetition_diagnostic(text):
    words=re.findall('\\w+',text.casefold())
    for period in range(3,min(128,len(words)//3)+1):
        for start in range(len(words)-3*period+1):
            block=words[start:start+period]
            if block==words[start+period:start+2*period]==words[start+2*period:start+3*period]:
                return {'period_words':period,'start_word':start}
    return None
answers=read_rows(RUN_DIR/'answers.jsonl');controls=read_rows(RUN_DIR/'greedy_controls.jsonl')
summary=[]
for strength in STRENGTHS:
    group=[r for r in answers if r['adapter_multiplier']==strength]
    summary.append({'multiplier':strength,'answers':len(group),'planned':8,
      'length_stops':sum(r['finish_reason']=='length' for r in group),
      'eos_stops':sum(r['finish_reason']=='eos' for r in group),
      'repetition_question_ids':[r['question_id'] for r in group if repetition_diagnostic(r['answer'])],
      'empty_answers':sum(not r['answer'].strip() for r in group)})
write_json(RUN_DIR/'technical_summary.json',{'status':metadata['status'],'by_scale':summary,
 'main_count':len(answers),'greedy_count':len(controls),
 'caution':'Post-hoc diagnostic development; content and safety are not inferred from these indicators.'})
report=['# Диагностика силы адаптера\n', 'Статус: '+metadata['status']+'\n',
 '|Множитель|Ответов / 8|Обрезано|Зациклилось|\n|---|---:|---:|---:|']
for r in summary:report.append(f"|{r['multiplier']}|{r['answers']}|{r['length_stops']}|{len(r['repetition_question_ids'])}|")
for qid in DIAGNOSTIC_IDS:
    report.extend(['\n## '+qid+'\n',question_index[qid]['question']+'\n'])
    for strength in STRENGTHS:
        group=[r for r in answers if r['question_id']==qid and r['adapter_multiplier']==strength]
        report.append('\n### Множитель '+str(strength)+'\n')
        if group:
            r=group[0];report.append(r['answer']+'\n')
            report.append(f"Остановка: {r['finish_reason']}; токенов: {r['generated_tokens']}; seed: {r['seed']}.\n")
        else:report.append('Результат отсутствует.\n')
report.append('\nЭто диагностический development. Технический PASS и отсутствие циклов сами по себе не подтверждают улучшение качества.\n')
(RUN_DIR/'comparison.md').write_text('\n'.join(report),encoding='utf-8')
ZIP_PATH=RUN_DIR.parent/(RUN_DIR.name+'.zip')
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file():archive.write(path,arcname=RUN_DIR.name+'/'+str(path.relative_to(RUN_DIR)))
print('Сохранено:',ZIP_PATH,'; статус:',metadata['status'])
from google.colab import files
files.download(str(ZIP_PATH))
